# Seaquest DQN â€” Member 2 Mid-Range Experiments
**Policy:** CnnPolicy | **Env:** ALE/Seaquest-v5

All outputs saved to `/kaggle/working/kerie/runs/`  
Download them manually after the notebook finishes via the **Output** tab.

In [ ]:
# !! IMPORTANT: Enable Internet first → Settings (right panel) → Internet → On
import subprocess, sys

for pkg in ['ale-py', 'gymnasium[atari]', 'stable-baselines3', 'opencv-python', 'tensorboard']:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        print(f'  OK: {pkg}')
    except Exception as e:
        print(f'  FAILED: {pkg}')
        print(f'  -> Enable Internet: Settings → Internet → On, then re-run this cell')
        raise

print('\nAll packages installed successfully.')

In [ ]:
import os, csv, gc
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
import ale_py
import torch

from stable_baselines3 import DQN
from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnNoModelImprovement

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
ENV_ID          = 'ALE/Seaquest-v5'
BUFFER_SIZE     = 50_000    # reduced from 100k — saves ~3 GB RAM
LEARNING_STARTS = 10_000
TARGET_UPDATE   = 1_000
N_STACK         = 4
OUTPUT_DIR      = '/kaggle/working/kerie/runs'

# With memory fixes we can now afford more steps
TIMESTEPS_DEFAULT = 500_000
TIMESTEPS_FULL    = 1_000_000

EXPERIMENTS = {
    1:  dict(name='exp1_baseline_mid',             learning_rate=0.0001, gamma=0.94, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.10, notes='Baseline mid — good balance'),
    2:  dict(name='exp2_lr_bump',                  learning_rate=0.0002, gamma=0.94, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.10, notes='Slight lr bump — faster policy updates'),
    3:  dict(name='exp3_higher_gamma',             learning_rate=0.0001, gamma=0.95, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.10, notes='Higher gamma — values future rewards more'),
    4:  dict(name='exp4_lower_eps_start',          learning_rate=0.0003, gamma=0.94, batch_size=64, exploration_initial_eps=0.9, exploration_final_eps=0.05, exploration_fraction=0.10, notes='Lower eps_start — less initial randomness'),
    5:  dict(name='exp5_lower_eps_end',            learning_rate=0.0001, gamma=0.96, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.02, exploration_fraction=0.10, notes='Lower eps_end — more greedy late stage'),
    6:  dict(name='exp6_faster_decay',             learning_rate=0.0002, gamma=0.95, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.05, notes='Faster decay — earlier shift to exploit'),
    7:  dict(name='exp7_slower_decay',             learning_rate=0.0001, gamma=0.94, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.25, notes='Slower decay — prolonged exploration'),
    8:  dict(name='exp8_mixed_mid',                learning_rate=0.0004, gamma=0.95, batch_size=64, exploration_initial_eps=0.9, exploration_final_eps=0.02, exploration_fraction=0.10, notes='Mixed mid — combined moderate changes'),
    9:  dict(name='exp9_moderate_decay_high_gamma',learning_rate=0.0003, gamma=0.96, batch_size=64, exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.15, notes='Moderate decay with higher gamma'),
    10: dict(name='exp10_best_of_mid',             learning_rate=0.0005, gamma=0.95, batch_size=64, exploration_initial_eps=0.9, exploration_final_eps=0.02, exploration_fraction=0.20, notes='Best-of-mid — final mid-range candidate ★ FULL 1M STEPS'),
}

print(f'Output: {OUTPUT_DIR}')
print(f'Exps 1-9 : {TIMESTEPS_DEFAULT:,} steps each')
print(f'Exp  10  : {TIMESTEPS_FULL:,} steps (full run)')
print(f'Buffer   : {BUFFER_SIZE:,} (optimize_memory_usage=True)')

In [ ]:
def make_env(render_mode=None):
    def _init():
        env = gym.make(ENV_ID, render_mode=render_mode)
        env = AtariWrapper(env)
        return env
    vec_env = DummyVecEnv([_init])
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)
    return vec_env

In [ ]:
# optimize_memory_usage explained:
#   DQN's replay buffer stores transitions: (state, action, reward, next_state).
#   But next_state of transition N is identical to state of transition N+1,
#   so by default SB3 stores every frame TWICE — wasting half the buffer RAM.
#   Setting optimize_memory_usage=True avoids this duplication by reconstructing
#   next_state on the fly from the neighboring entry, cutting buffer memory ~50%.
#   Trade-off: sampling is slightly slower, but we save ~1.5 GB with buffer=50k.
#
#   Note: optimize_memory_usage=True requires handle_timeout_termination=False
#   in the replay buffer. This is fine for Atari — AtariWrapper already handles
#   episode boundaries via life-loss detection, so timeout handling isn't needed.
#
#   Combined with gc.collect() + torch.cuda.empty_cache() between experiments,
#   this prevents the OOM crashes we hit when running 10 experiments sequentially.

def run_experiment(exp_id, total_timesteps=TIMESTEPS_DEFAULT):
    cfg     = EXPERIMENTS[exp_id]
    run_dir = os.path.join(OUTPUT_DIR, cfg['name'])

    print(f"\n{'='*60}")
    print(f"  Experiment {exp_id:02d}: {cfg['name']}")
    print(f"  Notes: {cfg['notes']}")
    print(f"  lr={cfg['learning_rate']}  gamma={cfg['gamma']}  batch={cfg['batch_size']}")
    print(f"  eps_start={cfg['exploration_initial_eps']}  eps_end={cfg['exploration_final_eps']}  eps_frac={cfg['exploration_fraction']}")
    print(f"{'='*60}")

    log_dir  = os.path.join(run_dir, 'tensorboard')
    eval_dir = os.path.join(run_dir, 'eval')
    os.makedirs(run_dir,  exist_ok=True)
    os.makedirs(log_dir,  exist_ok=True)
    os.makedirs(eval_dir, exist_ok=True)

    train_env = make_env()
    eval_env  = make_env()

    # Resume from checkpoint if it exists, otherwise create fresh model
    checkpoint_path = os.path.join(run_dir, 'dqn_model.zip')
    if os.path.isfile(checkpoint_path):
        print(f'  RESUMING from {checkpoint_path}')
        model = DQN.load(
            checkpoint_path,
            env = train_env,
            tensorboard_log = log_dir,
            verbose = 1,
        )
        reset_timesteps = False
    else:
        print(f'  Starting fresh')
        model = DQN(
            policy                  = 'CnnPolicy',
            env                     = train_env,
            learning_rate           = cfg['learning_rate'],
            gamma                   = cfg['gamma'],
            batch_size              = cfg['batch_size'],
            exploration_initial_eps = cfg['exploration_initial_eps'],
            exploration_final_eps   = cfg['exploration_final_eps'],
            exploration_fraction    = cfg['exploration_fraction'],
            buffer_size             = BUFFER_SIZE,
            learning_starts         = LEARNING_STARTS,
            target_update_interval  = TARGET_UPDATE,
            tensorboard_log         = log_dir,
            verbose                 = 1,
            optimize_memory_usage   = True,
            replay_buffer_kwargs    = {'handle_timeout_termination': False},
        )
        reset_timesteps = True

    early_stop = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=3, verbose=1
    )
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path = eval_dir,
        log_path             = eval_dir,
        eval_freq            = 25_000,
        n_eval_episodes      = 5,
        deterministic        = True,
        render               = False,
        callback_after_eval  = early_stop,
    )

    model.learn(
        total_timesteps     = total_timesteps,
        callback            = eval_callback,
        tb_log_name         = cfg['name'],
        reset_num_timesteps = reset_timesteps,
    )

    model_path = os.path.join(run_dir, 'dqn_model')
    model.save(model_path)
    print(f'\n  Model saved -> {model_path}.zip')

    train_env.close()
    eval_env.close()

    # Free memory between experiments — prevents OOM after ~8 sequential runs
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    eval_npz = os.path.join(eval_dir, 'evaluations.npz')
    if os.path.isfile(eval_npz):
        data         = np.load(eval_npz)
        timesteps    = data['timesteps']
        mean_rewards = data['results'].mean(axis=1)
        std_rewards  = data['results'].std(axis=1)

        plt.figure(figsize=(10, 5))
        plt.plot(timesteps, mean_rewards, label='Mean Reward')
        plt.fill_between(timesteps,
                         mean_rewards - std_rewards,
                         mean_rewards + std_rewards,
                         alpha=0.3, label='Std Dev')
        plt.xlabel('Timesteps')
        plt.ylabel('Eval Reward')
        plt.title(f"Exp {exp_id:02d}: {cfg['name']}\n"
                  f"lr={cfg['learning_rate']} gamma={cfg['gamma']} "
                  f"batch={cfg['batch_size']} eps_frac={cfg['exploration_fraction']}")
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plot_path = os.path.join(run_dir, 'reward_plot.png')
        plt.savefig(plot_path, dpi=150)
        plt.show()
        plt.close()
        print(f'  Plot saved  -> {plot_path}')

        csv_path = os.path.join(run_dir, 'results.csv')
        with open(csv_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['timestep', 'mean_reward', 'std_reward'])
            for t, m, s in zip(timesteps, mean_rewards, std_rewards):
                writer.writerow([int(t), f'{m:.2f}', f'{s:.2f}'])
        print(f'  CSV saved   -> {csv_path}')

    return os.path.join(run_dir, 'dqn_model.zip')

In [ ]:
# Exps 1-9 → 500k steps each | Exp 10 → 1M steps
# Finished experiments are skipped automatically on re-run.

results = {}
for exp_id in EXPERIMENTS:
    timesteps = TIMESTEPS_FULL if exp_id == 10 else TIMESTEPS_DEFAULT
    path = run_experiment(exp_id, total_timesteps=timesteps)
    results[exp_id] = path

print('ALL EXPERIMENTS COMPLETE')
for exp_id, path in results.items():
    print(f'  Exp {exp_id:02d}: {path}')
print('='*60)

In [ ]:
plt.figure(figsize=(14, 6))

for exp_id, cfg in EXPERIMENTS.items():
    eval_npz = os.path.join(OUTPUT_DIR, cfg['name'], 'eval', 'evaluations.npz')
    if not os.path.isfile(eval_npz):
        continue
    data         = np.load(eval_npz)
    timesteps    = data['timesteps']
    mean_rewards = data['results'].mean(axis=1)
    plt.plot(timesteps, mean_rewards, label=f"Exp{exp_id:02d} lr={cfg['learning_rate']} γ={cfg['gamma']}")

plt.xlabel('Timesteps')
plt.ylabel('Mean Eval Reward')
plt.title('All 10 Experiments — Mid-Range (Member 2, CnnPolicy)')
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
summary_path = os.path.join(OUTPUT_DIR, 'summary_all_experiments.png')
plt.savefig(summary_path, dpi=150)
plt.show()
print(f'Summary plot saved -> {summary_path}')

In [ ]:
print(f"{'Exp':<5} {'Name':<40} {'Best Reward':>12} {'Final Reward':>13}")
print('-' * 75)

summary_rows = []
for exp_id, cfg in EXPERIMENTS.items():
    eval_npz = os.path.join(OUTPUT_DIR, cfg['name'], 'eval', 'evaluations.npz')
    if not os.path.isfile(eval_npz):
        print(f"{exp_id:<5} {cfg['name']:<40} {'NOT DONE':>12}")
        continue
    data         = np.load(eval_npz)
    mean_rewards = data['results'].mean(axis=1)
    best_reward  = mean_rewards.max()
    final_reward = mean_rewards[-1]
    print(f"{exp_id:<5} {cfg['name']:<40} {best_reward:>12.1f} {final_reward:>13.1f}")
    summary_rows.append([exp_id, cfg['name'], cfg['learning_rate'], cfg['gamma'],
                         cfg['batch_size'], cfg['exploration_initial_eps'],
                         cfg['exploration_final_eps'], cfg['exploration_fraction'],
                         f'{best_reward:.1f}', f'{final_reward:.1f}', cfg['notes']])

summary_csv = os.path.join(OUTPUT_DIR, 'summary_table.csv')
with open(summary_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['exp', 'name', 'lr', 'gamma', 'batch', 'eps_start', 'eps_end', 'eps_frac', 'best_reward', 'final_reward', 'notes'])
    writer.writerows(summary_rows)
print(f'\nSummary CSV saved -> {summary_csv}')
print('\nDownload all outputs from the Output tab on the right ->')